# Yeu cau 4: Model Deployment
Trien khai FastAPI tren Colab dung ngrok. Load model tu MLflow Registry (diem cong).

In [ ]:
!pip install fastapi uvicorn pyngrok mlflow xgboost scikit-learn nest-asyncio -q

In [2]:
# Cau hinh ngrok
from pyngrok import ngrok, conf

NGROK_TOKEN = '3BYi01lJpaXOJeTc6sVxu433fJH_6MNQ3Ns3E6LsEkwHL6y8F'
ngrok.set_auth_token(NGROK_TOKEN)

In [3]:
import pickle
import mlflow
import mlflow.sklearn

# Load vectorizer
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

# Load model Production tu MLflow Registry (diem cong)
MODEL_NAME = 'SpamClassifier'
model = mlflow.sklearn.load_model(f'models:/{MODEL_NAME}/Production')
print('Da load model Production tu MLflow Registry!')

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Da load model Production tu MLflow Registry!


In [4]:
import re
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
import threading

nest_asyncio.apply()

app = FastAPI(title='Spam Classifier API')

class InputText(BaseModel):
    text: str

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

@app.get('/')
def root():
    return {'message': 'Spam Classifier API dang chay'}

@app.post('/predict')
def predict(input: InputText):
    cleaned = clean_text(input.text)
    vector  = tfidf.transform([cleaned])
    pred    = model.predict(vector)[0]
    label   = 'spam' if pred == 1 else 'ham'
    return {'text': input.text, 'predicted_label': label}

# Khoi dong server trong thread rieng
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000)

t = threading.Thread(target=run_server, daemon=True)
t.start()

# Tao ngrok tunnel
public_url = ngrok.connect(8000)
print(f'API public URL: {public_url}')
print(f'Test endpoint : {public_url}/predict')

INFO:     Started server process [15220]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


API public URL: NgrokTunnel: "https://bettie-hypoeutectic-maison.ngrok-free.dev" -> "http://localhost:8000"
Test endpoint : NgrokTunnel: "https://bettie-hypoeutectic-maison.ngrok-free.dev" -> "http://localhost:8000"/predict


INFO:     42.118.137.223:0 - "GET / HTTP/1.1" 200 OK
INFO:     42.118.137.223:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found


In [ ]:
# Test bang requests (thay URL bang public_url o tren)
import requests

PUBLIC_URL = public_url.public_url  # Lay tu cell tren

res1 = requests.post(f'{PUBLIC_URL}/predict', json={'text': 'FREE prize win money now click here'})
res2 = requests.post(f'{PUBLIC_URL}/predict', json={'text': 'Hi, are you free for lunch tomorrow?'})

print('Email 1 (spam?):', res1.json())
print('Email 2 (ham?) :', res2.json())

INFO:     171.250.184.79:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     171.250.184.79:0 - "POST /predict HTTP/1.1" 200 OK
Email 1 (spam?): {'text': 'FREE prize win money now click here', 'predicted_label': 'spam'}
Email 2 (ham?) : {'text': 'Hi, are you free for lunch tomorrow?', 'predicted_label': 'ham'}


INFO:     42.118.137.223:0 - "POST /predict HTTP/1.1" 200 OK
INFO:     42.118.137.223:0 - "POST /predict HTTP/1.1" 200 OK


## Test bang curl (chay tren terminal)

```bash
# Thay <PUBLIC_URL> bang URL tu ngrok
curl -X POST https://bettie-hypoeutectic-maison.ngrok-free.dev/predict -H 'Content-Type: application/json' -d '{"text": "FREE prize win now!"}'

Invoke-WebRequest -Uri "https://bettie-hypoeutectic-maison.ngrok-free.dev/predict" -Method POST -Headers @{"Content-Type"="application/json"} -Body '{"text": "Your message here"}'
# Ket qua mong doi:
# {"text": "FREE prize win now!", "predicted_label": "spam"}
```